In [55]:
import pandas as pd
from langchain.chat_models import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.embeddings.base import Embeddings
from langchain import PromptTemplate
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [56]:
class LlamaIndexEmbeddingAdapter(Embeddings):
    def __init__(self, llama_index_embedding):
        self.llama_index_embedding = llama_index_embedding

    def embed_documents(self, texts):
        return self.llama_index_embedding.get_text_embedding_batch(texts)

    def embed_query(self, text):
        return self.llama_index_embedding.get_text_embedding(text)

In [57]:
from dotenv import load_dotenv
load_dotenv()

True

In [58]:
llm =ChatOpenAI(temperature = 0, model = "gpt-3.5-turbo-16k")

In [59]:
xls_file=r'/Users/gizemkaryagdi/Desktop/Masaüstü/vector_database/vector_database/Project/QNA Yeni Gelen.xlsx'
data=pd.read_excel(xls_file,engine='openpyxl')

In [60]:
import pandas as pd
from langchain.schema import Document
from langchain.text_splitter import CharacterTextSplitter

df = pd.read_excel(xls_file)

def custom_text_splitter(row):
    content = f"soru1: {row['question']}\n"
    content += f"cevap1: {row['answer']}\n"
    
    for i in range(1, 21):
        query_col = f'query_{i}'
        if query_col in row and pd.notna(row[query_col]):
            content += f"soru1_alternatif_{i}: {row[query_col]}\n"
            content += f"cevap_1: {row['answer']}\n"
    
    return content

documents = []
for _, row in df.iterrows():
    content = custom_text_splitter(row)
    doc = Document(page_content=content, metadata={"qna_id": row['qna_id']})
    documents.append(doc)

text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

splits = text_splitter.split_documents(documents)

In [61]:
data.head()

,qna_id,question,answer,tags,query_1,query_2,query_3,query_4,query_5,query_6,...,query_11,query_12,query_13,query_14,query_15,query_16,query_17,query_18,query_19,query_20
0,1,Ders kitaplarına nereden ulaşabilirim?,<p>Öğrencilerimiz ders kitaplarının dijital ve...,NaN,Ders kitaplarına nasıl ulaşabilrim?,Ders kitaplarının dijitaline nasıl ulaşabilirim?,Ders kitabının PDF formatı nerede var?,Basılı kitapları nereden satın alabilirim?,Ders kitaplarını satın alabilir miyim?,Sınava gireceğimiz kitaplara nereden ulaşabilirim,...,kitap sipariş vereceğim,Ders metaryelerone nerden bakabiliri.,Kitaplar satılıyor mu,kitapsatis.anadolu.edu.tr,Açık öğretim ders notlarına erişmek istiyorum,ders özetleri lazim,kitap fiyatlarını öğrenmek istiyorum,ünite özetlerini indiremiyorum,ders kaynaklarına nasıl ulaşabilirim,PDF ders notlarına ulaşmak istiyorum
1,4,Tanıtım amaçlı derslerden sorumlu muyum?,"<p><a href=""https://ekampus.anadolu.edu.tr/"" t...",NaN,eKampüs sistemindeki Oryantasyon dersinden sor...,Diğer derslerim bölümünden sorumlu muyum?,e kampüste diğer derslerim bölümü diye bir şey...,e kampüste diğer derslerim bölümü nedir?,ekampüs diğer derslerimin sınavı ne zaman?,Tanıtım derslerinden sorumlu muyum?,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5,Mobil uygulama üzerinden sınav sonuçlarını nas...,"<p><a href=""https://mobil.anadolu.edu.tr"" targ...",NaN,Mobil uygulamada sınav sonuçlarımız görünecek ...,Mobil uygulamada sınav sonuçlarıma nasıl bakac...,Sınav sonuçlarım mobil uygulamada görünmüyor,Mobil uygulamada sınav sonuçları,Mobilde sınav sonuçları nerden göreceğim?,mobilden sınav sonuçlarımı görüntülemek istiyorum,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6,Anadolu Mobil Uygulamasını nasıl kullanabilirim?,"<p>Uygulama indiricisinden <a href=""https://mo...",NaN,Anadolu mobil uygulaması nedir?,Mobil uygulamayı nasıl indirebilirim,Mobil uygulamama nasıl girebilirim,Anadolu mobil uygulaması nereden indirebilirim,Mobil uygulamaya nasıl giriş yapabilirim?,Anadolu mobili kullanamıyorum,...,anadolu mobil hata alıyorum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8,Anadolu mobilden giriş yaptım cep telefonundan...,<p>Mobil cihazınıza uygun ePub okuyucu program...,NaN,ePub kitaplara nasıl erişebilirim?,Mobil uygulamadan ePub kitapları nereden göreb...,Mobilden EPUB kitabı nasıl açacağım?,epub kitapları uygulamadan nasıl görebilirim?,Mobil uygulamada Epub kitaplara nasıl erişirim,E pub kitaplara nereden ulaşırım?,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
llama_index_embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

/opt/anaconda3/envs/langchain/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [63]:
embed_model = LlamaIndexEmbeddingAdapter(llama_index_embed_model)

In [64]:
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say you don't know, don't try to make up an answer. 
Give answers in great detail. If the question is in English, respond in English. 
If the question is in Turkish, respond in Turkish. For other languages, respond in that language.

{context}

Question: {question}
Helpful Answer:"""
custom_rag_prompt = PromptTemplate.from_template(template)

In [65]:
def format_docs(doc):
    return "\n\n".join(doc.page_content for doc in doc)

In [66]:
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embed_model,
    persist_directory="/Users/gizemkaryagdi/Desktop/Masaüstü/vector_database/vector_database/Project/test10"
)
retriever = vectorstore.as_retriever()

In [67]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | custom_rag_prompt   | llm | StrOutputParser()
)

In [68]:
response = rag_chain.invoke("Anadolu Mobil Uygulamasını nasıl kullanabilirim?")
print(response)

Anadolu Mobil Uygulamasını kullanmak için uygulama indiricisinden Anadolu Mobil Uygulamasını indirmeniz gerekmektedir. Uygulamayı indirdikten sonra TC Kimlik numaranız ve şifrenizle giriş yapabilirsiniz. Uygulama üzerinden derslerinize erişebilir, sınav sonuçlarınızı görüntüleyebilir ve diğer öğrenci işlemlerinizi gerçekleştirebilirsiniz.
